# IHA Rota 

In [ ]:
import subprocess, sys

pkgs = ['pybullet', 'gymnasium>=0.29.0']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Kurulum tamamlandı.')

In [ ]:
import shutil, os, sys

input_root = '/kaggle/input'
WORK_DIR   = '/kaggle/working/iha_rota'
os.makedirs(WORK_DIR, exist_ok=True)

shutil.copytree(input_root, WORK_DIR, dirs_exist_ok=True)

for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f.endswith('.zip'):
            zip_path = os.path.join(root, f)
            shutil.unpack_archive(zip_path, root)
            os.remove(zip_path)
            print(f'Zip açıldı: {zip_path}')

project_root = None
for root, dirs, files in os.walk(WORK_DIR):
    if 'config.py' in files and 'env' in dirs:
        project_root = root
        break

if project_root is None:
    result = subprocess.run(['find', WORK_DIR, '-name', 'config.py'], capture_output=True, text=True)
    raise FileNotFoundError(
        f'config.py bulunamadı.\nWORK_DIR içeriği:\n{result.stdout}')

os.chdir(project_root)
sys.path.insert(0, project_root)
print('Proje kökü:', project_root)
print('Dosyalar:', sorted(os.listdir('.')))

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
import sys
sys.path.insert(0, '.')

from env.iha_env import IhaEnv
import numpy as np

env = IhaEnv(render_mode='headless', seed=0)
obs, _ = env.reset()
print(f'Ortam OK — obs boyutu: {obs.shape}')
print(f'Engel sayısı: {int(env.obstacle_grid.sum())}')

for _ in range(10):
    obs, rew, term, trunc, info = env.step(env.action_space.sample())

print(f'10 adım OK — drone konumu: {np.round(env.drone_pos, 2)}')
env.close()
print('Test geçti.')

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('train_module', 'train.py')
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

train_mod.main(visualize=False)

In [ ]:
import os
from config import MODEL_PATH, VERSION

if os.path.exists(MODEL_PATH):
    size_kb   = os.path.getsize(MODEL_PATH) / 1024
    full_path = os.path.abspath(MODEL_PATH)
    print(f'Model kaydedildi: {MODEL_PATH}  ({size_kb:.1f} KB)')
    print(f'İndirilecek konum: {full_path}')
else:
    print(f'Model kaydedilmedi — {VERSION} için henüz rekor kırılmadı.')